# Eye disease 10-class: model comparison (automatic loop)

Trains **4 light models + 5 heavy models** on the cleaned eye-disease split (`train.csv`, `val.csv`, `test.csv`), one after another, then fits an **SVM and a Random Forest** on frozen ResNet18 features.
Classification only. Original images only (no pre-made augmented images).

**No early stopping.** Every model trains for the full `EPOCHS_FULL` epochs. The checkpoint with the best **validation macro-F1** is kept, and the test set is used once at the end.

**Before running**
1. Add data -> your eye-disease dataset (the folder with the class folders such as `Glaucoma/`, `Healthy/`). The split CSVs are downloaded from GitHub automatically if they are not inside the dataset.
2. Accelerator **GPU T4 x2** (or P100), Internet **On** (pretrained weights and split CSVs).
3. First run with `SMOKE = True` (2 epochs, tiny data). If it finishes without errors, set `SMOKE = False` and use **Save & Run All**.

If a session dies, run the notebook again: finished models are skipped and unfinished ones resume from their last epoch.

Outputs: `results_bundle_seed42.zip` (probabilities, metrics, curves, confusion matrices).

In [ ]:
# STEP 1 - settings. SMOKE = True is the quick test. Set it to False for the real run.
import os, json, time, random, shutil, traceback, hashlib, urllib.request
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

SMOKE = True     # FIRST RUN: keep True (2 epochs, tiny). Then set False for the real run (full epochs, no early stopping).
SEEDS = [42]                 # add more seeds here if you want repeats, nothing else to edit
SEED = SEEDS[0]
EPOCHS_FULL = 50
BATCH = 32
USE_CLASS_WEIGHTS = True
RUN_LIGHT = True
RUN_HEAVY = True
RUN_CLASSICAL = True         # SVM + Random Forest on frozen ResNet18 features
LIGHT_MODELS = ["simple_cnn", "mobilenet_v2", "resnet18", "efficientnet_b0"]
HEAVY_MODELS = ["alexnet", "densenet121", "resnet50", "convnext_tiny", "swin_t"]

EPOCHS = 2 if SMOKE else EPOCHS_FULL
SMOKE_N = {"train": 20, "val": 6, "test": 10}     # images per class and split in the smoke test
IMG = 224                                         # all models see 224 x 224 images

def seed_everything(s):
    random.seed(s); np.random.seed(s)
    try:
        import torch
        torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    except Exception:
        pass
seed_everything(SEED)

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("work")
CACHE = Path("/kaggle/temp/cache") if Path("/kaggle/temp").exists() else WORK / "cache"
RES = RUNS = None
def use_seed(s):
    """point every output path at this seed's own folders, and seed the random generators"""
    global SEED, RES, RUNS
    SEED = s
    RES = WORK / f"results_seed{s}"; RUNS = WORK / f"runs_seed{s}"
    for d in (RES, RUNS, RES / "probs"): d.mkdir(parents=True, exist_ok=True)
    seed_everything(s)
for d in (WORK, CACHE): d.mkdir(parents=True, exist_ok=True)
use_seed(SEEDS[0])
print("seeds:", SEEDS, "| SMOKE:", SMOKE, "| epochs:", EPOCHS, "| early stopping: OFF | class weights:", USE_CLASS_WEIGHTS)

In [ ]:
# STEP 2 - find the images, read the split files, check everything before we spend GPU time
def find_data_root():
    if os.environ.get("EYE_DATA_ROOT"): return Path(os.environ["EYE_DATA_ROOT"])
    for base in [Path("/kaggle/input"), Path("/content"), Path(".")]:
        if base.exists():
            for folder in base.rglob("Glaucoma"):
                if folder.is_dir(): return folder.parent
    raise FileNotFoundError("Eye dataset not found. Add it in the 'Add data' panel (or set EYE_DATA_ROOT).")
DATA_ROOT = find_data_root(); print("images:", DATA_ROOT)

SPLIT_FILES = ["train.csv", "val.csv", "test.csv", "label_mapping.csv"]

def find_splits_dir():
    # A folder is only usable if every file we need is there.
    for folder in [DATA_ROOT / "splits", Path("data/splits"), WORK / "splits"]:
        if all((folder / name).is_file() for name in SPLIT_FILES): return folder
    folder = WORK / "splits"; folder.mkdir(parents=True, exist_ok=True)
    base = "https://raw.githubusercontent.com/nirjon001/eye-disease-classification/master/data/splits/"
    for name in SPLIT_FILES:
        if not (folder / name).is_file():
            print("downloading missing split file:", name)
            urllib.request.urlretrieve(base + name, folder / name)
    return folder
SPLITS = find_splits_dir(); print("splits:", SPLITS)

parts = []
for name in ["train", "val", "test"]:
    d = pd.read_csv(SPLITS / f"{name}.csv"); d["split"] = name; parts.append(d)
meta = pd.concat(parts, ignore_index=True)
need = {"path", "label", "group_id", "split"}
assert need <= set(meta.columns), f"missing columns: {need - set(meta.columns)}"

label_map = pd.read_csv(SPLITS / "label_mapping.csv").sort_values("label")
def short(name):
    return name.replace("Central Serous Chorioretinopathy [Color Fundus]", "CSCR").replace("[", "").replace("]", "")
SHORT = [short(c) for c in label_map["class"]]
K = len(SHORT); print("classes:", K, SHORT)
meta["y"] = meta["label"].astype(int)
meta["cls"] = [SHORT[i] for i in meta["y"]]

def resolve_path(rel):
    """CSV paths are relative to the image folder. Kaggle removes [ ] from folder names, so try that too."""
    p = DATA_ROOT / rel
    if p.is_file(): return p
    pp = Path(rel).parts
    return DATA_ROOT / " ".join(pp[0].replace("[", "").replace("]", "").split()) / Path(*pp[1:])
meta["src"] = meta["path"].map(lambda r: str(resolve_path(r)))
missing = [p for p in meta["src"] if not Path(p).exists()]
assert not missing, f"{len(missing)} files not found, e.g. {missing[:3]}"
print(f"all {len(meta)} files exist")

# safety checks (leakage rules)
assert meta.path.is_unique, "an image is listed twice"
g = meta.groupby("group_id")["split"].nunique()
assert (g == 1).all(), "a look-alike group is split across train/val/test"
assert set(meta.split) == {"train", "val", "test"}
if not SMOKE:
    n = meta.split.value_counts().to_dict(); print(n)
    assert (n["train"], n["val"], n["test"]) == (3038, 656, 657), "split sizes differ from the cleaned split"

if SMOKE:   # tiny subset, only to check that the code runs
    meta = pd.concat([meta[meta.split == s].groupby("cls").head(n) for s, n in SMOKE_N.items()], ignore_index=True)
print(pd.crosstab(meta.cls, meta.split))

In [ ]:
# STEP 3 - resize each image once to 256x256 (whole image, no cropping), saved in a cache (name = hash of the path)
def to_cache(args):
    rel, path = args
    out = CACHE / (hashlib.md5(rel.encode()).hexdigest() + ".jpg")
    if out.exists(): return str(out), 0
    with Image.open(path) as im:
        im.draft("RGB", (512, 512))                       # fast partial decode of the big 2004x1690 JPEGs
        im = ImageOps.exif_transpose(im).convert("RGB")
        im.resize((256, 256), Image.BILINEAR).save(out, quality=95)
    return str(out), 1
t0 = time.time()
with ThreadPoolExecutor(4) as ex:
    res = list(ex.map(to_cache, list(zip(meta.path, meta.src))))
meta["cache_path"] = [r[0] for r in res]
print(f"built {sum(r[1] for r in res)}, reused {sum(1 - r[1] for r in res)} in {time.time()-t0:.0f}s")
train_df = meta[meta.split == "train"].reset_index(drop=True)
val_df = meta[meta.split == "val"].reset_index(drop=True)
test_df = meta[meta.split == "test"].reset_index(drop=True)
print("train", len(train_df), "| val", len(val_df), "| test", len(test_df))

In [ ]:
# STEP 4 - data loading, augmentation, models
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
import torchvision.transforms.functional as TF
from torchvision.models import (alexnet, AlexNet_Weights, convnext_tiny, ConvNeXt_Tiny_Weights, densenet121, DenseNet121_Weights,
    efficientnet_b0, EfficientNet_B0_Weights, mobilenet_v2, MobileNet_V2_Weights, resnet18, ResNet18_Weights,
    resnet50, ResNet50_Weights, swin_t, Swin_T_Weights)
from sklearn.metrics import f1_score, confusion_matrix, classification_report
IMAGENET = ((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
mean, std = IMAGENET

train_tf = T.Compose([T.RandomResizedCrop(IMG, scale=(0.7, 1.0)), T.RandomHorizontalFlip(),
                      T.RandomRotation(15), T.ColorJitter(brightness=0.1, contrast=0.1),
                      T.ToTensor(), T.Normalize(mean, std)])                        # TRAINING only
eval_tf = T.Compose([T.Resize(IMG), T.ToTensor(), T.Normalize(mean, std)])          # val / test: resize + normalise only

class DS(Dataset):
    def __init__(self, df, tf): self.p, self.y, self.tf = df.cache_path.tolist(), df.y.tolist(), tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.tf(Image.open(self.p[i]).convert("RGB")), self.y[i]

class SimpleCNN(nn.Module):
    """4 conv blocks trained from scratch (the baseline)."""
    def __init__(self, n):
        super().__init__()
        def block(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, 128), block(128, 256))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(256, n))
    def forward(self, x): return self.head(self.features(x))

def build_model(name):
    """returns model, head params, body params. Pretrained models start from ImageNet weights; only the last layer is new."""
    if name == "simple_cnn":
        m = SimpleCNN(K); head = m.head
    elif name in ("resnet18", "resnet50"):
        m = (resnet18(weights=ResNet18_Weights.IMAGENET1K_V1) if name == "resnet18" else resnet50(weights=ResNet50_Weights.IMAGENET1K_V1))
        m.fc = nn.Linear(m.fc.in_features, K); head = m.fc
    elif name == "mobilenet_v2":
        m = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1); m.classifier[1] = nn.Linear(m.classifier[1].in_features, K); head = m.classifier[1]
    elif name == "alexnet":
        m = alexnet(weights=AlexNet_Weights.IMAGENET1K_V1); m.classifier[6] = nn.Linear(m.classifier[6].in_features, K); head = m.classifier[6]
    elif name == "efficientnet_b0":
        m = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1); m.classifier[1] = nn.Linear(m.classifier[1].in_features, K); head = m.classifier[1]
    elif name == "densenet121":
        m = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1); m.classifier = nn.Linear(m.classifier.in_features, K); head = m.classifier
    elif name == "convnext_tiny":
        m = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1); m.classifier[2] = nn.Linear(m.classifier[2].in_features, K); head = m.classifier[2]
    elif name == "swin_t":
        m = swin_t(weights=Swin_T_Weights.IMAGENET1K_V1); m.head = nn.Linear(m.head.in_features, K); head = m.head
    else:
        raise ValueError(f"unknown model: {name}")
    head_p = list(head.parameters()); ids = {id(p) for p in head_p}
    body_p = [p for p in m.parameters() if id(p) not in ids]
    return m, head_p, body_p

def topk_acc(probs, labels, k):
    top = np.argsort(-probs, axis=1)[:, :k]
    return float(np.mean([l in row for l, row in zip(labels, top)]))

def metrics_from_probs(probs, labels):
    preds = probs.argmax(1)
    return dict(acc=float((preds == np.array(labels)).mean()), top3=topk_acc(probs, labels, 3), top5=topk_acc(probs, labels, 5),
                macro_f1=float(f1_score(labels, preds, labels=list(range(K)), average="macro", zero_division=0)))

# augmentation preview: one image through each augmentation
img = Image.open(train_df.cache_path.iloc[0]).convert("RGB")
variants = [("original", img), ("rotate 15", TF.rotate(img, 15)), ("flip", TF.hflip(img)),
            ("zoom", T.RandomResizedCrop(256, scale=(0.5, 0.6))(img)),
            ("brightness", T.ColorJitter(brightness=(1.3, 1.3))(img)), ("contrast", T.ColorJitter(contrast=(1.3, 1.3))(img))]
fig, axes = plt.subplots(1, len(variants), figsize=(3 * len(variants), 3.4))
for ax, (n, im) in zip(axes, variants): ax.imshow(im); ax.set_title(n); ax.axis("off")
plt.tight_layout(); plt.savefig(RES / "augmentation_grid.png", dpi=100); plt.show()
print("device:", "cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# STEP 5 - the training function: FULL epochs (no early stopping), checkpoint every epoch, resumes after a crash
def predict_probs(model, loader, device):
    model.eval(); P, Y = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", enabled=(device == "cuda")):
                out = model(x)
            P.append(out.float().softmax(1).cpu().numpy()); Y += y.tolist()
    return np.concatenate(P), Y

def atomic_save(obj, path):
    tmp = str(path) + ".tmp"; torch.save(obj, tmp); os.replace(tmp, path)

def fit(name, tr_df, va_df, te_df, run_dir, epochs):
    run_dir = Path(run_dir); run_dir.mkdir(parents=True, exist_ok=True)
    if (run_dir / "DONE").exists():
        prev = json.load(open(run_dir / "result.json"))
        if prev.get("epochs") == epochs:
            print("already done:", run_dir); return prev
        # leftover from a different run (e.g. the 2-epoch smoke test): never reuse it as a real result
        print(f"STALE run in {run_dir}: it trained {prev.get('epochs')} epochs but this run wants {epochs} -> deleting and retraining")
        shutil.rmtree(run_dir); run_dir.mkdir(parents=True, exist_ok=True)
    seed_everything(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model, head, body = build_model(name)
    model = model.to(device).to(memory_format=torch.channels_last)
    lr_head, lr_body = (1e-3, 1e-3) if name == "simple_cnn" else (1e-3, 1e-4)      # pretrained body: smaller learning rate
    opt = torch.optim.AdamW([{"params": body, "lr": lr_body}, {"params": head, "lr": lr_head}], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    try: scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
    except Exception: scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))
    if USE_CLASS_WEIGHTS:
        cnt = np.bincount(tr_df.y, minlength=K).astype(float); w = len(tr_df) / (K * np.maximum(cnt, 1))
        loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=device), label_smoothing=0.1)
    else:
        loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
    nw = 4 if device == "cuda" else 0
    tl = DataLoader(DS(tr_df, train_tf), batch_size=BATCH, shuffle=True, num_workers=nw, pin_memory=(device == "cuda"), drop_last=len(tr_df) > BATCH)
    vl = DataLoader(DS(va_df, eval_tf), batch_size=64, num_workers=nw)
    start, best_f1, best_ep, hist = 0, -1.0, -1, []
    if (run_dir / "last.pt").exists():
        ck = torch.load(run_dir / "last.pt", map_location="cpu", weights_only=False)   # our own file
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        scaler.load_state_dict(ck["scaler"]); start, best_f1, best_ep, hist = ck["epoch"] + 1, ck["best_f1"], ck["best_ep"], ck["hist"]
        print(f"resuming {name} from epoch {start}")
    for ep in range(start, epochs):                       # every epoch runs: there is no early stopping
        t0 = time.time(); model.train(); tot, n, correct = 0.0, 0, 0
        for x, y in tl:
            x, y = x.to(device, non_blocking=True).to(memory_format=torch.channels_last), y.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", enabled=(device == "cuda")):
                out = model(x); loss = loss_fn(out, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item() * len(y); n += len(y); correct += (out.argmax(1) == y).sum().item()
        sched.step()
        vp, vy = predict_probs(model, vl, device)
        vm = metrics_from_probs(vp, vy)
        vloss = float(-np.mean(np.log(np.maximum(vp[np.arange(len(vy)), vy], 1e-9))))
        rec = dict(epoch=ep + 1, train_loss=tot / n, train_acc=correct / n, val_loss=vloss, val_acc=vm["acc"], val_f1=vm["macro_f1"], seconds=time.time() - t0)
        hist.append(rec); open(run_dir / "metrics.jsonl", "a").write(json.dumps(rec) + "\n")
        if vm["macro_f1"] > best_f1:                      # best epoch is chosen on VALIDATION macro-F1
            best_f1, best_ep = vm["macro_f1"], ep + 1; atomic_save(model.state_dict(), run_dir / "best.pt")
        atomic_save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(), scaler=scaler.state_dict(),
                         epoch=ep, best_f1=best_f1, best_ep=best_ep, hist=hist), run_dir / "last.pt")
        print(f"[{name}] ep {ep+1}/{epochs} loss {rec['train_loss']:.3f} acc {rec['train_acc']:.3f} | val acc {vm['acc']:.3f} F1 {vm['macro_f1']:.3f} | {rec['seconds']:.1f}s")
    # curves
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
    ax[0].plot([h["train_loss"] for h in hist], label="train"); ax[0].plot([h["val_loss"] for h in hist], label="val"); ax[0].set_title("loss"); ax[0].legend()
    ax[1].plot([h["train_acc"] for h in hist], label="train acc"); ax[1].plot([h["val_acc"] for h in hist], label="val acc"); ax[1].plot([h["val_f1"] for h in hist], label="val macro-F1"); ax[1].set_title("accuracy / F1"); ax[1].legend()
    for a in ax: a.set_xlabel("epoch")
    plt.tight_layout(); plt.savefig(run_dir / "curves.png", dpi=110); plt.close()
    result = dict(model=name, params=sum(p.numel() for p in model.parameters()), epochs=epochs, best_epoch=best_ep, best_val_f1=best_f1,
                  best_val_acc=max(h["val_acc"] for h in hist), sec_per_epoch=float(np.mean([h["seconds"] for h in hist])),
                  train_images=len(tr_df), val_images=len(va_df), class_weights=USE_CLASS_WEIGHTS, seed=SEED)
    model.load_state_dict(torch.load(run_dir / "best.pt", map_location=device))   # best epoch chosen on VALIDATION
    vp, vy = predict_probs(model, vl, device)
    np.save(run_dir / "val_probs.npy", vp); np.save(run_dir / "val_labels.npy", np.array(vy))
    tel = DataLoader(DS(te_df, eval_tf), batch_size=64, num_workers=nw)             # TEST: once, with the best checkpoint
    tp, ty = predict_probs(model, tel, device)
    tm = metrics_from_probs(tp, ty)
    result.update(test_acc=tm["acc"], test_top3=tm["top3"], test_top5=tm["top5"], test_f1=tm["macro_f1"])
    np.save(run_dir / "test_probs.npy", tp); np.save(run_dir / "test_labels.npy", np.array(ty))
    preds = tp.argmax(1)
    json.dump(classification_report(ty, preds, labels=list(range(K)), target_names=SHORT, output_dict=True, zero_division=0), open(run_dir / "test_report.json", "w"), indent=1)
    cm = confusion_matrix(ty, preds, labels=list(range(K)))
    fig, ax = plt.subplots(figsize=(8, 7)); ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(K)); ax.set_xticklabels(SHORT, rotation=45, ha="right"); ax.set_yticks(range(K)); ax.set_yticklabels(SHORT)
    for i in range(K):
        for j in range(K): ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8, color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"{name} - test confusion matrix")
    plt.tight_layout(); plt.savefig(run_dir / "confusion_matrix.png", dpi=110); plt.close()
    json.dump(result, open(run_dir / "result.json", "w"), indent=1)
    (run_dir / "last.pt").unlink(missing_ok=True)            # keep best.pt only (saves disk)
    open(run_dir / "DONE", "w").close()
    print("RESULT", json.dumps({k: (round(v, 4) if isinstance(v, float) else v) for k, v in result.items()}))
    del model; torch.cuda.empty_cache()
    return result

FAILED = {}
def run_models(names, group):
    for name in names:
        try:
            fit(name, train_df, val_df, test_df, RUNS / "main" / name, EPOCHS)
        except Exception as e:                               # one bad model must not stop the others
            FAILED[name] = f"{type(e).__name__}: {e}"
            print(f"!!! {name} FAILED -> {FAILED[name]}"); traceback.print_exc()
            torch.cuda.empty_cache()
    json.dump(FAILED, open(RES / "failed_models.json", "w"), indent=1)
    print("failed so far:", FAILED if FAILED else "none")

In [ ]:
# STEP 6 - classical baselines: SVM + Random Forest on frozen ResNet18 features (512 numbers per image)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

def run_classical():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    fe = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1); fe.fc = nn.Identity(); fe = fe.to(device).eval()
    def feats(df):
        out = []
        with torch.no_grad():
            for x, _ in DataLoader(DS(df, eval_tf), batch_size=64, num_workers=0):
                out.append(fe(x.to(device)).float().cpu().numpy())
        return np.concatenate(out)
    Xtr, Xva, Xte = feats(train_df), feats(val_df), feats(test_df)
    ytr, yva, yte = train_df.y.to_numpy(), val_df.y.to_numpy(), test_df.y.to_numpy()
    sc = StandardScaler().fit(Xtr); Str, Sva, Ste = sc.transform(Xtr), sc.transform(Xva), sc.transform(Xte)
    best, best_f1 = None, -1
    for C in [1, 10, 100]:                                    # C chosen on the validation set
        svm = SVC(kernel="rbf", C=C, class_weight="balanced", random_state=SEED).fit(Str, ytr)
        f1 = f1_score(yva, svm.predict(Sva), average="macro", zero_division=0); print(f"SVM C={C}: val macro-F1 {f1:.3f}")
        if f1 > best_f1: best, best_f1 = svm, f1
    rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=SEED).fit(Xtr, ytr)
    rows = []
    for name, model, X in [("svm_resnet18_feat", best, Ste), ("random_forest_resnet18_feat", rf, Xte)]:
        p = model.predict(X)
        rows.append(dict(model=name, test_acc=float((p == yte).mean()), test_f1=float(f1_score(yte, p, labels=list(range(K)), average="macro", zero_division=0))))
        print(rows[-1])
    del fe; torch.cuda.empty_cache()
    return rows

In [ ]:
# STEP 7 - summary table + one zip per seed (built from saved files, nothing typed by hand)
def summarize(seed, classical_rows=None):
    rows = [json.load(open(p)) for p in sorted((RUNS / "main").glob("*/result.json"))]
    assert rows, f"no model finished. Failed: {FAILED}"
    tab = pd.DataFrame(rows)
    tab["params_M"] = (tab.params / 1e6).round(2)
    cols = ["model", "params_M", "epochs", "best_epoch", "best_val_f1", "test_acc", "test_top3", "test_top5", "test_f1", "sec_per_epoch"]
    tab = tab[cols].sort_values("test_f1", ascending=False).round(3)
    if classical_rows:
        tab = pd.concat([tab, pd.DataFrame(classical_rows).round(3)], ignore_index=True).sort_values("test_f1", ascending=False)
    tab.to_csv(RES / "summary_table.csv", index=False); print(tab.to_string(index=False))
    ax = tab.set_index("model")[["test_acc", "test_f1"]].plot(kind="barh", figsize=(9, 5)); ax.invert_yaxis(); ax.set_xlim(0, 1)
    ax.set_title(f"Test-set comparison (seed {seed})"); plt.tight_layout(); plt.savefig(RES / "model_comparison.png", dpi=120); plt.show()
    json.dump(sorted(FAILED) if FAILED else [], open(RES / "failed_models.json", "w"))
    for name in [m for m in tab.model if (RUNS / "main" / m).exists()]:
        for f in ["curves.png", "confusion_matrix.png", "test_report.json", "val_probs.npy", "val_labels.npy", "test_probs.npy", "test_labels.npy", "result.json"]:
            s = RUNS / "main" / name / f
            if s.exists(): shutil.copy(s, RES / "probs" / f"{name}_{f}" if f.endswith(".npy") else RES / f"{name}_{f}")
    shutil.copy(SPLITS / "train.csv", RES / "train_split_used.csv")
    shutil.make_archive(str(WORK / f"results_bundle_seed{seed}"), "zip", RES)
    print("zipped:", WORK / f"results_bundle_seed{seed}.zip", "| failed models:", FAILED if FAILED else "none")
    print("model checkpoints (best.pt) stay in", RUNS / "main", "- download them from the Output tab if you need them")

In [ ]:
# STEP 8 - the automatic loop: light models, then heavy models, then SVM / Random Forest.
# A failure is logged, not fatal. A finished model is skipped; an unfinished one resumes.
ALL_FAILED = {}
for s in SEEDS:
    use_seed(s); FAILED.clear()
    print("\n" + "=" * 70 + f"\nSEED {s}\n" + "=" * 70)
    if RUN_LIGHT: run_models(LIGHT_MODELS, "light")
    if RUN_HEAVY: run_models(HEAVY_MODELS, "heavy")
    classical = run_classical() if RUN_CLASSICAL else None
    ALL_FAILED[s] = dict(FAILED)
    summarize(s, classical)
print("\nALL SEEDS DONE. failed:", ALL_FAILED)